# Milestone 4 — Which Images Compress Well?

Test rank and compression across artificial image patterns and rotations.

In [ ]:
import os
repo_path="/content/svd-image-compression"
if os.path.isdir(repo_path):
    !git -C /content/svd-image-compression pull --ff-only
else:
    !git clone -q https://github.com/billygoattrain/svd-image-compression.git /content/svd-image-compression
%cd /content/svd-image-compression

In [ ]:
import sys, pathlib
from skimage import data
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
# Works after downloading/extracting the package, or from the project root in Colab.
project_root = pathlib.Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
from compress import rank_k_approx, storage, compression_ratio, rel_error, predicted_error, k_for_error
(project_root / "figures").mkdir(exist_ok=True)
photo_path = project_root / "data" / "my_photo.png"
if photo_path.exists():
    P = np.array(Image.open(photo_path).convert("L"), dtype=float)
else:
    P = None
    print("No personal photo yet: add data/my_photo.png before final submission.")

## Build five test patterns

In [ ]:
N=256
i,j=np.meshgrid(np.arange(N),np.arange(N),indexing="ij")
def stripes(angle,width=16):
 t=np.deg2rad(angle)
 return 255.0*(np.floor((i*np.cos(t)+j*np.sin(t))/width)%2)
rng=np.random.default_rng(0)
patterns={"stripes 0deg":stripes(0),"stripes 45deg":stripes(45),"checkerboard":255.0*((i//32+j//32)%2),"disk":255.0*((i-N/2)**2+(j-N/2)**2<(N/3)**2),"noise":rng.integers(0,256,size=(N,N)).astype(float)}
fig,axs=plt.subplots(1,5,figsize=(15,3))
for ax,(name,M) in zip(axs,patterns.items()):ax.imshow(M,cmap="gray");ax.set_title(name);ax.axis("off")
plt.tight_layout();plt.show()

**Predictions made before comparing ranks:** I expect axis-aligned stripes to have rank 1 and the checkerboard to have rank 2, because their rows repeat only one or two patterns. I expect rotated stripes and the disk to require many independent row patterns; random noise should be the hardest and have full rank. Thus I predict straight stripes will compress best, and random noise worst. The measured results below will test these predictions.

## How SVD measures the five patterns

For each synthetic image, the full matrix rank counts independent patterns. The smallest `k` giving no more than 5% relative Frobenius error measures how many SVD terms must be retained; the idealized compression ratio counts numerical entries, not saved PNG bytes.

## Numerical comparison

In [ ]:
for name,M in patterns.items():
    ss=np.linalg.svd(M,compute_uv=False);k5=k_for_error(ss,0.05)
    print(f"{name:18s} rank={np.linalg.matrix_rank(M):3d} k5={k5:3d} ratio={compression_ratio(N,N,k5):.2f}")
print("Tilted image top-left corner:\n", stripes(45)[:4,:12]/255)


A tilted image has shifted row patterns; many rows are not scalar multiples of one another.

In [ ]:
fig,axes=plt.subplots(1,len(patterns),figsize=(15,3.5))
for ax,(name,M) in zip(axes,patterns.items()):
    Mk=np.clip(rank_k_approx(M,5),0,255)
    ax.imshow(Mk,cmap="gray",vmin=0,vmax=255)
    ax.set_title(f"{name}\nk=5");ax.axis("off")
plt.tight_layout();plt.show()

## Angles figure

In [ ]:
angles=np.arange(0,91,5)
needed=[k_for_error(np.linalg.svd(stripes(float(a)),compute_uv=False),0.05) for a in angles]
plt.plot(angles,needed,"o-");plt.xlabel("Angle (degrees)");plt.ylabel("k for 5% error")
plt.title("Same stripes, different angles");plt.grid(True);plt.tight_layout()
plt.savefig(project_root/"figures"/"angles.png",dpi=150);plt.show()
print("k at 0°, 5°, 45°, 90°:",needed[0],needed[1],needed[9],needed[-1])
print("Change in 5%-error factor storage from 0° to 5°:",needed[1]/needed[0])

Tilting the stripes raises the rank needed to achieve the same approximation error.

## Color extension

In [ ]:
C=data.astronaut().astype(float)
C50=np.stack([rank_k_approx(C[:,:,c],50) for c in range(3)],axis=2)
print("RGB shape:",C.shape)
print("RGB relative error at rank 50/channel",rel_error(C,C50))
print("Storage grayscale at k=50:",storage(512,512,50))
print("Storage color at k=50 (3 channels):",3*storage(512,512,50))
plt.imshow(np.clip(C50,0,255).astype(np.uint8));plt.title("RGB image, rank 50 per channel");plt.axis("off");plt.show()

Storage for three separate color channels is three times the grayscale factor storage at the same k.

## Exercise 1 — Why do tilted stripes have higher rank?

The 45° pattern shifts horizontally from row to row. Because successive row patterns are shifted rather than scalar multiples of each other, many independent rows are needed to represent the image.

In [ ]:
print("45° stripes, top-left 4 rows by 12 columns (0=black, 1=white):")
print(stripes(45)[:4,:12]/255)
print("Numerical rank of 0° and 45° patterns:",np.linalg.matrix_rank(stripes(0)),np.linalg.matrix_rank(stripes(45)))

## Exercise 2 — Two similar-looking images with very different compressibility

**Prediction before running:** Both designs resemble horizontal black-and-white stripes. The clean pattern should be rank 1 and easy to compress. Adding random grain without changing the basic bands should look similar at a glance but require many more SVD terms for a stringent 5% error tolerance.

In [ ]:
clean_pattern=stripes(0)
grain_rng=np.random.default_rng(42)
grainy_pattern=np.clip(clean_pattern+grain_rng.normal(0,40,size=(N,N)),0,255)
fig,axes=plt.subplots(1,2,figsize=(9,4))
for ax,M,name in zip(axes,[clean_pattern,grainy_pattern],["Clean stripes","Grainy stripes"]):
    ax.imshow(M,cmap="gray",vmin=0,vmax=255);ax.set_title(name);ax.axis("off")
plt.tight_layout();plt.show()
for name,M in [("Clean",clean_pattern),("Grainy",grainy_pattern)]:
    sv=np.linalg.svd(M,compute_uv=False)
    k=k_for_error(sv,0.05)
    print(f"{name}: matrix rank={np.linalg.matrix_rank(M)}, k for 5% error={k}, theoretical ratio={compression_ratio(N,N,k):.2f}x")

**Interpretation:** Check the printed ranks above. Even though both images contain the same visible horizontal bands, the added pixel-level grain creates many independent details, making an accurate low-rank reconstruction harder. The strict numerical tolerance is sensitive to that grain.

## Exercise 3 — Color image: three separately compressed channels

The experiment above compresses each of the astronaut image's red, green, and blue channels at rank 50, clips them to [0,255], and stacks the results into one RGB image. The idealized numerical storage is three times that of a grayscale image of the same width and height at the same rank, because each channel needs separate SVD factors. It can still be less than the original RGB pixel count.

## Exercise 4 — Why text can look unreadable despite low relative error

The text image contains a large, fairly uniform background, and approximations can preserve those many background values while smearing thin letter strokes. Frobenius error averages numerical differences across the image; it does not specifically measure whether a person can read the letters. Visual inspection of the rank-5 reconstruction is therefore essential.

## One-sentence finding for the README

Tilting a 256×256 striped image by only 5° raised the SVD rank needed for at most 5% reconstruction error from **1 to about 69**, multiplying the number of factor values by about **69**, even though the picture still consists of stripes.

## What we found — review in your own words

Straight, axis-aligned stripes compressed best because their row patterns repeat, while random noise and tilted shapes needed many more singular components to preserve small visual details. The checkerboard also had low rank despite looking complicated. Looking at the rank-5 reconstructions confirmed that numerical rank reflects repeated matrix patterns rather than visual complexity. The angle experiment was especially revealing: small tilts made the same stripe structure much harder for SVD to reproduce accurately.

**Review:** Confirm the actual 5° result printed above and personalize the explanation to reflect your own observations before submission.